In [ ]:
import numpy as np
import pandas as pd
import rasterio as rio
from scipy.optimize import curve_fit
from scipy.stats import t
from matplotlib import pyplot as plt
from tqdm.autonotebook import tqdm
from copy import deepcopy

C:\Users\Administrator\AppData\Local\Temp\ipykernel_26464\1640231034.py:7: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
names = locals()

In [3]:
country_raster_1km_    = rio.open('../results/country-rasters/country-raster-eck4-1km.tif')
country_raster_50km_   = rio.open('../results/country-rasters/country-raster-eck4-50km.tif')
mining_suitability_1km = rio.open('../results/mining-suitability/global-mining-suitability-1km.tif').read(1)
available_area_BAU_1km = rio.open('../results/mining-area/available-area-BAU-1km.tif').read(1)
available_area_CON_1km = rio.open('../results/mining-area/available-area-CON-1km.tif').read(1)
country_raster_50km    = country_raster_50km_.read(1)

In [4]:
def get_index(i, j, src_res='50km', dst_res='1km'):
    scale_dict = {'10km': 10, '50km': 50}
    scale = scale_dict[src_res]

    src_ref = rio.open(f'../results/country-rasters/country-raster-eck4-{src_res}.tif')
    dst_ref = rio.open(f'../results/country-rasters/country-raster-eck4-{dst_res}.tif')

    scale = 50
    a = int(abs(dst_ref.transform[5] - src_ref.transform[5]) / 1e3) 
    b = int(abs(dst_ref.transform[2] - src_ref.transform[2]) / 1e3) 

    # print(a, b)

    row_min = i * scale - a
    row_max = (i + 1) * scale - a
    col_min = j * scale - b
    col_max = (j + 1) * scale - b

    return row_min, row_max, col_min, col_max

In [5]:
def allocate_demand_to_extraction(scenario, level_1, level_2, level_3, max_r, res='50km'):
    country_raster_ = rio.open(f'../results/country-rasters/country-raster-eck4-{res}.tif')
    economy_raster_ = rio.open(f'../results/economy-rasters/economy-raster-eck4-{res}.tif')
    country_raster  = country_raster_.read(1)
    economy_raster  = economy_raster_.read(1)

    if scenario == 'CON':
        s1 = 'BAU'
    elif scenario == 'IAP':
        s1 = 'CE'
    else:
        s1 = scenario

    sand_demand_raster      = rio.open(f'../results/uncertainty-analysis/sand-demand-2023-2050-{s1}-{res}-{level_1}.tif').read(1)
    gravel_demand_raster    = rio.open(f'../results/uncertainty-analysis/gravel-demand-2023-2050-{s1}-{res}-{level_1}.tif').read(1)
    limestone_demand_raster = rio.open(f'../results/uncertainty-analysis/limestone-demand-2023-2050-{s1}-{res}-{level_1}.tif').read(1)

    total_demand_raster     = sand_demand_raster + gravel_demand_raster + limestone_demand_raster
    global_aggregate_demand = np.sum(total_demand_raster) / 1e9  # Gt


    if scenario in ['CON', 'IAP']:
        extraction_capacity = rio.open(f'../results/uncertainty-analysis/extraction-capacity-{scenario}-{res}-{level_2}{level_3}.tif').read(1)
    else:
        extraction_capacity = rio.open(f'../results/uncertainty-analysis/extraction-capacity-BAU-{res}-{level_2}{level_3}.tif').read(1)


    final_extraction_raster = np.zeros(country_raster.shape, dtype=np.float64)
    remained_demand_global  = np.zeros(country_raster.shape, dtype=np.float64)
    remained_capacity_global = np.zeros(country_raster.shape, dtype=np.float64)

    country_codes = np.unique(country_raster)
    country_codes = country_codes[country_codes > 0]


    for code in tqdm(country_codes, desc="Allocating demand by country"):
        mask = (country_raster == code)
        demand_raster   = total_demand_raster * mask / 1e6  # Mt
        capacity_raster = extraction_capacity * mask

        demand_raster_list     = [deepcopy(demand_raster)]
        capacity_raster_list   = [deepcopy(capacity_raster)]
        extraction_raster_list = []

        for r in range(max_r+1):
            demand_raster   = deepcopy(demand_raster_list[-1])
            capacity_raster = deepcopy(capacity_raster_list[-1])

            if r == 0:
                extraction_raster = np.where(demand_raster > capacity_raster, capacity_raster, demand_raster)
                remained_capacity = capacity_raster - extraction_raster
                remained_demand   = demand_raster - extraction_raster

            else:
                remained_demand   = deepcopy(demand_raster)
                remained_capacity = deepcopy(capacity_raster)
                extraction_raster = np.zeros_like(demand_raster, dtype=np.float64)

                i_set, j_set = np.where(demand_raster > 0)
                for i, j in zip(i_set, j_set):
                    if not mask[i, j]:  
                        continue
                    a_demand = demand_raster[i, j]

                    i_min, i_max = max(0, i-r), min(mask.shape[0], i+r+1)
                    j_min, j_max = max(0, j-r), min(mask.shape[1], j+r+1)
                    submask = mask[i_min:i_max, j_min:j_max]
                    around_capacity = remained_capacity[i_min:i_max, j_min:j_max] * submask

                    total_capacity_around = np.sum(around_capacity)
                    if total_capacity_around > 0:
                        weight_array = around_capacity / total_capacity_around
                        an_extraction = min(a_demand, total_capacity_around)
                        extraction_around = weight_array * an_extraction
                        extraction_raster[i_min:i_max, j_min:j_max] += extraction_around
                        remained_capacity[i_min:i_max, j_min:j_max] -= extraction_around
                        remained_demand[i, j] -= an_extraction

            extraction_raster_list.append(extraction_raster)
            demand_raster_list.append(remained_demand)
            capacity_raster_list.append(remained_capacity)


        final_extraction_raster += np.sum(extraction_raster_list, axis=0)
        remained_demand_global  += remained_demand
        remained_capacity_global += remained_capacity

    coverage_1 = 100 * np.sum(final_extraction_raster/1e3) / global_aggregate_demand
    # print(f"✅ Global coverage ({scenario}-{level_1}{level_2}{level_3}): {np.sum(final_extraction_raster)/1e3:.3f}/{global_aggregate_demand:.3f} Gt ({coverage_1:.1f}%)")

    for code in country_codes:
        mask = (country_raster == code)
        country_remained_demand   = remained_demand_global[mask].sum()
        country_remained_capacity = remained_capacity_global[mask].sum()

        if country_remained_demand == 0:
            continue

        if country_remained_capacity > 0:
            # print(code)
            cap_country = remained_capacity_global[mask]
            w_country = cap_country / cap_country.sum()
            add_local = w_country * min(country_remained_demand, country_remained_capacity)

            final_extraction_raster[mask]  += add_local
            remained_capacity_global[mask] -= add_local

            country_remained_demand -= add_local.sum()

        if country_remained_demand == 0:
            remained_demand_global[mask] = 0

        # if country_remained_demand > 0:
        #     print('gg', code)
    
    remained_capacity_global = np.where(economy_raster > 0, remained_capacity_global, 0)
    w_global = remained_capacity_global / remained_capacity_global.sum()
    aaa = (np.sum(total_demand_raster) - np.sum(final_extraction_raster) * 1e6) / 1e6
    # final_extraction_raster += w_global * np.sum(remained_demand_global)
    final_extraction_raster += w_global * aaa 


    coverage_2 = 100 * np.sum(final_extraction_raster/1e3) / global_aggregate_demand
    print(f"✅ Global coverage ({scenario}-{level_1}{level_2}{level_3}): {np.sum(final_extraction_raster)/1e3:.3f}/{global_aggregate_demand:.3f} Gt ({coverage_2:.1f}%)")


    output_path = f'../results/uncertainty-analysis/allocated-extraction-2023-2050-{scenario}-{res}-{level_1}{level_2}{level_3}.tif'
    with rio.open(
        output_path,
        'w',
        driver='GTiff',
        height=country_raster.shape[0],
        width=country_raster.shape[1],
        count=1,
        dtype=final_extraction_raster.dtype,
        crs=country_raster_.crs,
        transform=country_raster_.transform,
        compress='lzw',
    ) as dst:
        dst.write(final_extraction_raster, 1)

    return final_extraction_raster, remained_demand_global

In [ ]:
for scenario in ['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CCR', 'CE']:
    for l_1 in ['L', 'M', 'H']:
        for l_2 in ['L', 'M', 'H']:
            for l_3 in ['L', 'M', 'H']:
                gg = allocate_demand_to_extraction(scenario, l_1, l_2, l_3, 10)